# TechArena CTR: history-weighted CatBoost GPU
Attach both private prepared TechArena datasets. Same frozen 1600-tree recipe as GPU001; the only fitting change is importance weighting by supplied causal history length.
Exact chronological label alignment is verified before fitting. Both date gates and saved-model replay are required; 0.83 is not guaranteed.


In [ ]:
from pathlib import Path
import importlib.util, hashlib
import torch
assert torch.cuda.is_available(), "Enable Kaggle GPU"
roots = list(Path("/kaggle/input").rglob("bundle_manifest.json"))
assert len(roots) == 1, "Attach the original prepared TechArena dataset"
DATA_ROOT = roots[0].parent
source_path = Path("/kaggle/working/catboost_gpu_weighted_runner.py")
source_path.write_text("\"\"\"Fixed-capacity CatBoost GPU expert on the existing causal feature bundle.\r\n\r\nNo evaluation labels enter fitting or early stopping. Main day6 is followed by\r\nunchanged day5 confirmation only if its conditional gain gate passes.\r\n\"\"\"\r\nimport gc\r\nimport importlib.util\r\nimport json\r\nimport time\r\nfrom pathlib import Path\r\n\r\nimport catboost\r\nimport numpy as np\r\nfrom catboost import CatBoostClassifier, Pool\r\n\r\nPARAMS=dict(iterations=1600,depth=8,learning_rate=.04,l2_leaf_reg=10.,\r\n            loss_function=\"Logloss\",task_type=\"GPU\",devices=\"0\",border_count=63,\r\n            bootstrap_type=\"Bernoulli\",subsample=.8,random_seed=6801,\r\n            boosting_type=\"Plain\",thread_count=4,allow_writing_files=False,\r\n            verbose=100)\r\nBLEND=.25\r\n\r\n\r\ndef helpers(root):\r\n    spec=importlib.util.spec_from_file_location(\"ctr_cb_gpu_helpers\",Path(root)/\"runner.py\")\r\n    module=importlib.util.module_from_spec(spec)\r\n    spec.loader.exec_module(module)\r\n    return module\r\n\r\n\r\ndef predict(model,features):\r\n    out=[]\r\n    for off in range(0,len(features),25000):\r\n        x=features.block(off,min(off+25000,len(features)))\r\n        out.append(model.predict_proba(x,thread_count=4)[:,1])\r\n    return np.concatenate(out)\r\n\r\n\r\ndef training_weights(neural_root, day, labels, h):\r\n    \"\"\"Label-free target history distribution, with exact fitting-row alignment.\"\"\"\r\n    neural_root=Path(neural_root)\r\n    nm=json.loads((neural_root/\"neural_manifest.json\").read_text())\r\n    for name in [\"events.npz\", \"val5.npz\", \"val6.npz\"]:\r\n        assert h.digest(neural_root/name)==nm[\"files\"][name][\"sha256\"]\r\n    with np.load(neural_root/\"events.npz\") as z:\r\n        timestamp=z[\"timestamp\"].copy()\r\n        target=z[\"target\"].copy()\r\n        bins=np.digitize((z[\"hist\"]>=0).sum(1),[0,4,9,19],right=True)\r\n    shares=[]\r\n    for d in [5,6]:\r\n        with np.load(neural_root/f\"val{d}.npz\") as z:\r\n            b=np.repeat(bins[z[\"indices\"]],10)\r\n            w=z[\"weights\"]\r\n            share=np.bincount(b,weights=w,minlength=5)/w.sum()\r\n            shares.append(share)\r\n    assert np.allclose(shares[0],shares[1],atol=1e-7)\r\n    target_share=shares[0]\r\n    event_day=(timestamp-int(nm[\"start\"]))//86400\r\n    selected=(event_day>=day-3)&(event_day<day)\r\n    expected=(target[selected,None]==np.arange(10)[None,:]).astype(\"float32\").ravel()\r\n    assert np.array_equal(expected,labels), \"Fitting labels and event chronology differ\"\r\n    assert (timestamp[selected]<int(nm[\"start\"])+day*86400).all()\r\n    b=np.repeat(bins[selected],10)\r\n    source_share=np.bincount(b,minlength=5)/len(b)\r\n    assert (source_share>0).all()\r\n    weights=(target_share/source_share)[b].astype(\"float32\")\r\n    weights/=weights.mean()\r\n    assert np.isfinite(weights).all() and (weights>0).all()\r\n    return weights,dict(target_history_share=target_share.tolist(),\r\n                        training_history_share=source_share.tolist(),\r\n                        weight_min=float(weights.min()),weight_max=float(weights.max()),\r\n                        alignment=\"exact chronological labels; strictly pre-cutoff events\",\r\n                        population=\"unlabeled evaluation history distribution reconstructed from fixed weights\")\r\n\r\n\r\ndef fit(root,output,manifest,day,h,neural_root):\r\n    started=time.monotonic()\r\n    output.mkdir(parents=True,exist_ok=False)\r\n    days=list(range(day-3,day))\r\n    validation=manifest[\"validation\"][str(day)]\r\n    specs=[manifest[\"train\"][str(d)] for d in days]\r\n    labels=[]\r\n    for spec in specs:\r\n        meta=json.loads((root/spec[\"metadata\"]).read_text())\r\n        assert meta[\"last_timestamp\"]<validation[\"cutoff\"]\r\n        labels.append(np.load(root/spec[\"labels\"]))\r\n    y=np.concatenate(labels).astype(\"float32\")\r\n    weights,weight_recipe=training_weights(neural_root,day,y,h)\r\n    feature=h.FeatureBlocks(root,[s[\"features\"] for s in specs])\r\n    assert len(feature)==len(y)\r\n    recipe=dict(history_importance=weight_recipe,parameters=PARAMS,blend_weight=BLEND,day=day,fit_days=days,\r\n                fit_rows=len(y),columns=354,validation_cutoff=validation[\"cutoff\"],\r\n                input_manifest_sha256=h.digest(root/\"bundle_manifest.json\"),\r\n                source_sha256=h.digest(Path(__file__)),catboost=catboost.__version__,\r\n                selection=\"fixed before scoring; no eval_set or early stopping\")\r\n    (output/\"recipe.json\").write_text(json.dumps(recipe,indent=2))\r\n    temporary=output/\"fit_features.npy\"\r\n    assert temporary.resolve().parent==output.resolve()\r\n    matrix=np.lib.format.open_memmap(temporary,mode=\"w+\",dtype=\"float32\",shape=(len(y),354))\r\n    for off in range(0,len(y),50000):\r\n        end=min(off+50000,len(y));matrix[off:end]=feature.block(off,end)\r\n        if off%500000==0:print(dict(stage=\"prepare_features\",day=day,rows=end,total=len(y)),flush=True)\r\n    matrix.flush()\r\n    pool=Pool(matrix,label=y,weight=weights,thread_count=4)\r\n    model=CatBoostClassifier(**PARAMS)\r\n    print(dict(stage=\"catboost_gpu_fit\",day=day,rows=len(y),iterations=PARAMS[\"iterations\"]),flush=True)\r\n    model.fit(pool)\r\n    model_path=output/\"model.cbm\"\r\n    model.save_model(str(model_path))\r\n    del pool\r\n    matrix._mmap.close()\r\n    del matrix,y,weights\r\n    gc.collect()\r\n    # Remove only the scratch array authored by this fresh experiment.\r\n    assert temporary.resolve().parent==output.resolve()\r\n    temporary.unlink()\r\n    with np.load(root/validation[\"data\"]) as z:\r\n        val={k:z[k].copy() for k in z.files}\r\n    xv=h.FeatureBlocks(root,[validation[\"features\"]])\r\n    assert len(xv)==len(val[\"y\"]) and len(xv)%10==0\r\n    raw=predict(model,xv)\r\n    assert np.isfinite(raw).all() and ((raw>=0)&(raw<=1)).all()\r\n    pred=h.rank((1-BLEND)*h.rank(val[\"baseline\"])+BLEND*h.rank(raw))\r\n    m=h.metrics(val[\"y\"],pred,val[\"baseline\"],val[\"weights\"])\r\n    restored=CatBoostClassifier()\r\n    restored.load_model(str(model_path))\r\n    replay=predict(restored,xv)\r\n    replay_error=float(np.max(np.abs(replay-raw)))\r\n    assert replay_error<1e-12,replay_error\r\n    m.update(day=day,raw_auc=float(h.roc_auc_score(val[\"y\"],raw)),rows=len(raw),\r\n             saved_model_replay_max_error=replay_error,saved_checkpoint_replay=\"PASS\",\r\n             model_sha256=h.digest(model_path),elapsed_seconds=round(time.monotonic()-started,1),\r\n             official_auc=None,csv_ready=False)\r\n    np.savez_compressed(output/\"predictions.npz\",raw=raw,pred=pred,**val)\r\n    (output/\"report.json\").write_text(json.dumps(m,indent=2))\r\n    print(json.dumps(m,indent=2),flush=True)\r\n    return m\r\n\r\n\r\ndef run(root,output,neural_root):\r\n    root,output=Path(root),Path(output)\r\n    h=helpers(root)\r\n    manifest=h.load_bundle(root,verify=True)\r\n    output.mkdir(parents=True,exist_ok=False)\r\n    (output/\"catboost_gpu_weighted_runner.py\").write_text(Path(__file__).read_text())\r\n    reports=[fit(root,output/\"day6\",manifest,6,h,neural_root)]\r\n    if reports[0][\"positive_gate\"]:\r\n        reports.append(fit(root,output/\"day5\",manifest,5,h,neural_root))\r\n    eligible=len(reports)==2 and all(m[\"positive_gate\"] for m in reports)\r\n    summary=dict(reports=reports,eligible_for_review=eligible,target=.83,\r\n                 target_achieved=bool(eligible and reports[0][\"history_adjusted_auc\"]>=.83),\r\n                 csv_ready=False,official_auc=None,\r\n                 note=\"Same reused main dates; fixed history-weighted CatBoost GPU expert. Local metric, not official forecast.\")\r\n    (output/\"summary.json\").write_text(json.dumps(summary,indent=2))\r\n    return summary\r\n", encoding="utf-8")
assert hashlib.sha256(source_path.read_bytes()).hexdigest() == "133e47d002c174d6f15c090f2fffc7f0ef63e4b287f3cc6b1d9a30b02e7b2d9c"
spec = importlib.util.spec_from_file_location("ctr_cb_weighted", source_path)
cb_weighted = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cb_weighted)
neural_manifests = list(Path("/kaggle/input").rglob("neural_manifest.json"))
assert len(neural_manifests)==1, neural_manifests
NEURAL_ROOT=neural_manifests[0].parent
RUN_NAME="ctr-catboost-gpu-002-weighted"
output=Path("/kaggle/working")/RUN_NAME
print("Frozen parameters:",cb_weighted.PARAMS)
summary=cb_weighted.run(DATA_ROOT,output,NEURAL_ROOT)
summary

In [ ]:
from IPython.display import FileLink, display
from pathlib import Path
import json, zipfile
print("Experiment:", RUN_NAME)
report = json.loads((output / "summary.json").read_text())
for result in report["reports"]:
    print("Day", result["day"], "Adjusted AUC:", round(result["history_adjusted_auc"], 9), "Ordinary AUC:", round(result["ordinary_auc"], 9), "Gate:", result["positive_gate"])
print("Target 0.83 reached:", report["target_achieved"])
archive = Path("/kaggle/working") / (RUN_NAME + "-results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for path in output.rglob("*"):
        if path.is_file():
            z.write(path, path.relative_to(output))
display(FileLink(str(archive)))
